1.Data Ingestion

In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("ChicagoCrime_BigData_Project") \
    .config("spark.sql.shuffle.partitions", "400") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .getOrCreate()

spark

In [0]:
from pyspark.sql.types import *

crime_schema = StructType([
    StructField("ID", LongType(), True),
    StructField("Case_Number", StringType(), True),
    StructField("Date", StringType(), True),
    StructField("Block", StringType(), True),
    StructField("IUCR", StringType(), True),
    StructField("Primary_Type", StringType(), True),
    StructField("Description", StringType(), True),
    StructField("Location_Description", StringType(), True),
    StructField("Arrest", BooleanType(), True),
    StructField("Domestic", BooleanType(), True),
    StructField("Beat", IntegerType(), True),
    StructField("District", IntegerType(), True),
    StructField("Ward", IntegerType(), True),
    StructField("Community_Area", IntegerType(), True),
    StructField("FBI_Code", StringType(), True),
    StructField("X_Coordinate", DoubleType(), True),
    StructField("Y_Coordinate", DoubleType(), True),
    StructField("Year", IntegerType(), True),
    StructField("Updated_On", StringType(), True),
    StructField("Latitude", DoubleType(), True),
    StructField("Longitude", DoubleType(), True),
    StructField("Location", StringType(), True)
])

In [0]:
crime_raw_df = spark.read \
    .option("header", "true") \
    .schema(crime_schema) \
    .csv("dbfs:/Volumes/workspace/crime_data/crime/Crimes_-_2001_to_Present_20260226.csv")

crime_raw_df.printSchema()

root
 |-- ID: long (nullable = true)
 |-- Case_Number: string (nullable = true)
 |-- Date: string (nullable = true)
 |-- Block: string (nullable = true)
 |-- IUCR: string (nullable = true)
 |-- Primary_Type: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Location_Description: string (nullable = true)
 |-- Arrest: boolean (nullable = true)
 |-- Domestic: boolean (nullable = true)
 |-- Beat: integer (nullable = true)
 |-- District: integer (nullable = true)
 |-- Ward: integer (nullable = true)
 |-- Community_Area: integer (nullable = true)
 |-- FBI_Code: string (nullable = true)
 |-- X_Coordinate: double (nullable = true)
 |-- Y_Coordinate: double (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Updated_On: string (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- Location: string (nullable = true)



In [0]:
crime_raw_df.count()


8502217

In [0]:
crime_raw_df.select("Year").distinct().orderBy("Year").show()

+----+
|Year|
+----+
|2001|
|2002|
|2003|
|2004|
|2005|
|2006|
|2007|
|2008|
|2009|
|2010|
|2011|
|2012|
|2013|
|2014|
|2015|
|2016|
|2017|
|2018|
|2019|
|2020|
+----+
only showing top 20 rows


In [0]:
crime_raw_df.select("Arrest").groupBy("Arrest").count().show()

+------+-------+
|Arrest|  count|
+------+-------+
|  true|2137809|
| false|6364408|
+------+-------+



In [0]:
from pyspark.sql.functions import col, count, when

crime_raw_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in crime_raw_df.columns
]).show()

+---+-----------+----+-----+----+------------+-----------+--------------------+------+--------+----+--------+------+--------------+--------+------------+------------+----+----------+--------+---------+--------+
| ID|Case_Number|Date|Block|IUCR|Primary_Type|Description|Location_Description|Arrest|Domestic|Beat|District|  Ward|Community_Area|FBI_Code|X_Coordinate|Y_Coordinate|Year|Updated_On|Latitude|Longitude|Location|
+---+-----------+----+-----+----+------------+-----------+--------------------+------+--------+----+--------+------+--------------+--------+------------+------------+----+----------+--------+---------+--------+
|  0|          0|   0|    0|   0|           0|          0|               15642|     0|       0|   0|      47|614818|        613685|       0|       94705|       94705|   0|         0|   94705|    94705|   94705|
+---+-----------+----+-----+----+------------+-----------+--------------------+------+--------+----+--------+------+--------------+--------+------------+---

In [0]:
crime_clean_df = crime_raw_df.dropna(
    subset=["Arrest", "Primary_Type", "District", "Year"]
)

crime_clean_df.count()

8502170

In [0]:
crime_clean_df = crime_clean_df.dropDuplicates(["ID"])

In [0]:
from pyspark.sql.functions import to_timestamp

crime_clean_df = crime_clean_df.withColumn(
    "Date",
    to_timestamp(col("Date"), "MM/dd/yyyy hh:mm:ss a")
)

In [0]:
from pyspark.sql.functions import hour, month, dayofweek

crime_clean_df = crime_clean_df \
    .withColumn("Hour", hour(col("Date"))) \
    .withColumn("Month", month(col("Date"))) \
    .withColumn("DayOfWeek", dayofweek(col("Date"))) \
    .withColumn("IsWeekend", (dayofweek(col("Date")).isin([1,7])).cast("integer"))

In [0]:
spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA crime_data")

DataFrame[]

In [0]:
crime_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/Volumes/workspace/crime_data/crime/Crimes_-_2001_to_Present_20260226.csv")

In [0]:
from pyspark.sql.functions import col

# Replace spaces with underscore and remove special characters
crime_df_clean = crime_df

for column in crime_df.columns:
    new_column = column.strip() \
                       .replace(" ", "_") \
                       .replace("-", "_") \
                       .replace("/", "_") \
                       .replace(".", "_")
    crime_df_clean = crime_df_clean.withColumnRenamed(column, new_column)

crime_df_clean.printSchema()

root
 |-- ID: integer (nullable = true)
 |-- Case_Number: string (nullable = true)
 |-- Date: string (nullable = true)
 |-- Block: string (nullable = true)
 |-- IUCR: string (nullable = true)
 |-- Primary_Type: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Location_Description: string (nullable = true)
 |-- Arrest: boolean (nullable = true)
 |-- Domestic: boolean (nullable = true)
 |-- Beat: integer (nullable = true)
 |-- District: integer (nullable = true)
 |-- Ward: integer (nullable = true)
 |-- Community_Area: integer (nullable = true)
 |-- FBI_Code: string (nullable = true)
 |-- X_Coordinate: integer (nullable = true)
 |-- Y_Coordinate: integer (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Updated_On: string (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- Location: string (nullable = true)



In [0]:
crime_df_clean.write \
    .mode("overwrite") \
    .partitionBy("Year") \
    .saveAsTable("crime_parquet_table")

In [0]:
from pyspark.sql.functions import broadcast

# Example: small lookup table (district statistics)
district_counts = crime_clean_df.groupBy("District").count()

# Broadcast join
crime_joined_df = crime_clean_df.join(
    broadcast(district_counts),
    on="District",
    how="left"
)

In [0]:
crime_df = crime_df.repartition(400, "Year")

2.Feature Engineering

In [0]:
from pyspark.sql.functions import when

crime_df = spark.table("crime_parquet_table")

crime_df = crime_df.withColumn(
    "label",
    when(crime_df.Arrest == True, 1).otherwise(0)
)

crime_df.select("Arrest", "label").show(5)

+------+-----+
|Arrest|label|
+------+-----+
| false|    0|
|  true|    1|
| false|    0|
| false|    0|
| false|    0|
+------+-----+
only showing top 5 rows


In [0]:
from pyspark.sql.functions import col, to_timestamp, hour, month, dayofweek

crime_df = spark.table("crime_parquet_table")

# Convert Date to timestamp (if needed)
crime_df = crime_df.withColumn(
    "Date",
    to_timestamp(col("Date"), "MM/dd/yyyy hh:mm:ss a")
)

# Create time features
crime_df = crime_df \
    .withColumn("Hour", hour(col("Date"))) \
    .withColumn("Month", month(col("Date"))) \
    .withColumn("DayOfWeek", dayofweek(col("Date"))) \
    .withColumn("IsWeekend", (dayofweek(col("Date")).isin([1,7])).cast("integer"))

crime_df.printSchema()

root
 |-- ID: integer (nullable = true)
 |-- Case_Number: string (nullable = true)
 |-- Date: timestamp (nullable = true)
 |-- Block: string (nullable = true)
 |-- IUCR: string (nullable = true)
 |-- Primary_Type: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Location_Description: string (nullable = true)
 |-- Arrest: boolean (nullable = true)
 |-- Domestic: boolean (nullable = true)
 |-- Beat: integer (nullable = true)
 |-- District: integer (nullable = true)
 |-- Ward: integer (nullable = true)
 |-- Community_Area: integer (nullable = true)
 |-- FBI_Code: string (nullable = true)
 |-- X_Coordinate: integer (nullable = true)
 |-- Y_Coordinate: integer (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Updated_On: string (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- Location: string (nullable = true)
 |-- Hour: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- DayOfWeek: int

In [0]:
selected_columns = [
    "Year",
    "Month",
    "Hour",
    "DayOfWeek",
    "District",
    "Beat",
    "Community_Area",
    "Primary_Type",
    "Domestic",
    "IsWeekend",
    "label"
]

crime_ml_df = crime_df.select(selected_columns)

crime_ml_df.printSchema()

root
 |-- Year: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- Hour: integer (nullable = true)
 |-- DayOfWeek: integer (nullable = true)
 |-- District: integer (nullable = true)
 |-- Beat: integer (nullable = true)
 |-- Community_Area: integer (nullable = true)
 |-- Primary_Type: string (nullable = true)
 |-- Domestic: boolean (nullable = true)
 |-- IsWeekend: integer (nullable = true)
 |-- label: integer (nullable = false)



In [0]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline

# Index categorical feature
primary_indexer = StringIndexer(
    inputCol="Primary_Type",
    outputCol="Primary_Type_index",
    handleInvalid="keep"
)

# One-hot encode
primary_encoder = OneHotEncoder(
    inputCol="Primary_Type_index",
    outputCol="Primary_Type_vec"
)

# Assemble all features
assembler = VectorAssembler(
    inputCols=[
        "Year",
        "Month",
        "Hour",
        "DayOfWeek",
        "District",
        "Beat",
        "Community_Area",
        "Domestic",
        "IsWeekend",
        "Primary_Type_vec"
    ],
    outputCol="features"
)

In [0]:
from pyspark.sql.functions import col

train_df = crime_ml_df.filter(col("Year") < 2022)
test_df = crime_ml_df.filter(col("Year") >= 2022)

print("Train Count:", train_df.count())
print("Test Count:", test_df.count())

Train Count: 7477221
Test Count: 1024996


In [0]:
crime_ml_df = crime_ml_df.dropna()

In [0]:
crime_ml_df = crime_ml_df.dropna(subset=[
    "Year",
    "Month",
    "Hour",
    "DayOfWeek",
    "District",
    "Beat",
    "Community_Area",
    "Domestic",
    "IsWeekend",
    "Primary_Type",
    "label"
])

In [0]:
train_df = crime_ml_df.filter(col("Year") < 2021)
val_df   = crime_ml_df.filter((col("Year") >= 2021) & (col("Year") < 2022))
test_df  = crime_ml_df.filter(col("Year") >= 2022)

print("Train:", train_df.count())
print("Validation:", val_df.count())
print("Test:", test_df.count())

Train: 6653918
Validation: 209613
Test: 1024954


In [0]:
from pyspark.sql.functions import col

train_df = crime_ml_df.filter(col("Year") < 2021)
val_df   = crime_ml_df.filter((col("Year") >= 2021) & (col("Year") < 2022))
test_df  = crime_ml_df.filter(col("Year") >= 2022)

print("Train:", train_df.count())
print("Validation:", val_df.count())
print("Test:", test_df.count())

Train: 6653918
Validation: 209613
Test: 1024954


3.Model Training

In [0]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline

# Index categorical column
primary_indexer = StringIndexer(
    inputCol="Primary_Type",
    outputCol="Primary_Type_index",
    handleInvalid="keep"
)

# One-hot encode
primary_encoder = OneHotEncoder(
    inputCol="Primary_Type_index",
    outputCol="Primary_Type_vec"
)

# Assemble features
assembler = VectorAssembler(
    inputCols=[
        "Year",
        "Month",
        "Hour",
        "DayOfWeek",
        "District",
        "Beat",
        "Community_Area",
        "Domestic",
        "IsWeekend",
        "Primary_Type_vec"
    ],
    outputCol="features",
    handleInvalid="keep"
)

# Logistic Regression model
lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=20
)

# Define pipeline (THIS was missing)
pipeline_lr = Pipeline(stages=[
    primary_indexer,
    primary_encoder,
    assembler,
    lr
])

In [0]:
lr_model = pipeline_lr.fit(train_df)

In [0]:
predictions_lr = lr_model.transform(test_df)

In [0]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    metricName="areaUnderROC"
)

auc_lr = evaluator.evaluate(predictions_lr)

print("Logistic Regression AUC:", auc_lr)

Logistic Regression AUC: 0.781963544237379


In [0]:
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=50,
    maxDepth=10
)

pipeline_rf = Pipeline(stages=[
    primary_indexer,
    primary_encoder,
    assembler,
    rf
])

In [0]:
rf_model = pipeline_rf.fit(train_df)

In [0]:
predictions_rf = rf_model.transform(test_df)

In [0]:
auc_rf = evaluator.evaluate(predictions_rf)
print("Random Forest AUC:", auc_rf)

Random Forest AUC: 0.7834056520240911


In [0]:
from pyspark.ml.classification import DecisionTreeClassifier

dt = DecisionTreeClassifier(
    featuresCol="features",
    labelCol="label",
    maxDepth=10
)

pipeline_dt = Pipeline(stages=[
    primary_indexer,
    primary_encoder,
    assembler,
    dt
])

In [0]:
dt_model = pipeline_dt.fit(train_df)

In [0]:
predictions_dt = dt_model.transform(test_df)

In [0]:
auc_dt = evaluator.evaluate(predictions_dt)
print("Decision Tree AUC:", auc_dt)

Decision Tree AUC: 0.4070756792484828


In [0]:
print("Model Comparison (AUC)")
print("------------------------")
print("Logistic Regression:", auc_lr)
print("Random Forest:", auc_rf)
print("Decision Tree:", auc_dt)

Model Comparison (AUC)
------------------------
Logistic Regression: 0.781963544237379
Random Forest: 0.7834056520240911
Decision Tree: 0.4070756792484828


In [0]:
best_model = rf_model

In [0]:
best_model = lr_model

In [0]:
best_rf = rf_model.stages[-1]
importances = best_rf.featureImportances

print(importances)

(42,[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,23,24,25,26,27,28,29,30,31,32,35,36,37],[0.003819607641484299,3.6244410011870376e-05,0.006627112971310863,1.121025126632378e-05,0.0010007872819466817,0.0008623881053372285,0.0012078583895170961,0.00857728474560852,1.0774165055764074e-05,0.05800691077824992,0.0032559848786332447,0.03756302000496195,0.6564271716078907,0.002374014238606676,0.0033258406018411854,0.016999989772548293,0.011132648934442057,0.004457990493140532,0.0027378131056384493,0.06907159816813954,0.02849694813181287,0.04851070145993139,0.010568873866590788,1.0954073726409244e-05,0.00014966961007044233,0.010933453235175793,0.006587822660148463,0.006827272263521246,5.120025648271871e-07,0.0003035135808333583,3.8359477482450877e-07,1.9694689998352825e-05,2.2445352263807607e-06,6.859850437782936e-05,1.3107245615580956e-05])


In [0]:
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

paramGrid = ParamGridBuilder() \
    .addGrid(rf.numTrees, [20, 50]) \
    .addGrid(rf.maxDepth, [5, 10]) \
    .build()

In [0]:
crossval = CrossValidator(
    estimator=pipeline_rf,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator,
    numFolds=3,
    parallelism=2
)

In [0]:
import os
os.environ["SPARKML_TEMP_DFS_PATH"] = "/Volumes/workspace/crime_data/ml_temp"

In [0]:
cv_model = crossval.fit(train_df)

In [0]:
cv_predictions = cv_model.transform(test_df)
auc_cv = evaluator.evaluate(cv_predictions)

print("Tuned Random Forest AUC:", auc_cv)

Tuned Random Forest AUC: 0.7834318629651427


In [0]:
spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA crime_data")

DataFrame[]

In [0]:
spark.sql("SHOW VOLUMES IN workspace.crime_data").show()

+----------+-----------+
|  database|volume_name|
+----------+-----------+
|crime_data|      crime|
|crime_data|    ml_temp|
+----------+-----------+



In [0]:
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.crime_data.ml_models")

DataFrame[]

In [0]:
cv_model.bestModel.write().overwrite().save(
    "/Volumes/workspace/crime_data/ml_models/best_rf_model"
)

In [0]:
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline

rf2 = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=50,
    maxDepth=10
)

pipeline_rf2 = Pipeline(stages=[
    primary_indexer,
    primary_encoder,
    assembler,
    rf2
])

In [0]:
spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA crime_data")

crime_df = spark.table("crime_parquet_table")

In [0]:
from pyspark.sql.functions import col, to_timestamp, hour, month, dayofweek, when

crime_df = crime_df.withColumn(
    "Date",
    to_timestamp(col("Date"), "MM/dd/yyyy hh:mm:ss a")
)

crime_df = crime_df \
    .withColumn("Hour", hour(col("Date"))) \
    .withColumn("Month", month(col("Date"))) \
    .withColumn("DayOfWeek", dayofweek(col("Date"))) \
    .withColumn("IsWeekend", (dayofweek(col("Date")).isin([1,7])).cast("integer"))

crime_df = crime_df.withColumn(
    "label",
    when(col("Arrest") == True, 1).otherwise(0)
)

selected_columns = [
    "Year","Month","Hour","DayOfWeek",
    "District","Beat","Community_Area",
    "Primary_Type","Domestic","IsWeekend","label"
]

crime_ml_df = crime_df.select(selected_columns).dropna()

In [0]:
train_df = crime_ml_df.filter(col("Year") < 2022)
test_df = crime_ml_df.filter(col("Year") >= 2022)

In [0]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline

primary_indexer = StringIndexer(
    inputCol="Primary_Type",
    outputCol="Primary_Type_index",
    handleInvalid="keep"
)

primary_encoder = OneHotEncoder(
    inputCol="Primary_Type_index",
    outputCol="Primary_Type_vec"
)

assembler = VectorAssembler(
    inputCols=[
        "Year","Month","Hour","DayOfWeek",
        "District","Beat","Community_Area",
        "Domestic","IsWeekend","Primary_Type_vec"
    ],
    outputCol="features",
    handleInvalid="keep"
)

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=50,
    maxDepth=10
)

pipeline_rf = Pipeline(stages=[
    primary_indexer,
    primary_encoder,
    assembler,
    rf
])

In [0]:
small_train_df = train_df.sample(fraction=0.2, seed=42)

In [0]:
spark.conf.set("spark.sql.shuffle.partitions", 200)

start = time.time()
train_df.groupBy("District").count().collect()
end = time.time()

print("200 partitions:", end - start)

200 partitions: 5.38273286819458


In [0]:
spark.conf.set("spark.sql.shuffle.partitions", 400)

import time
start = time.time()

train_df.groupBy("District").count().collect()

end = time.time()
print("400 partitions:", end - start)

400 partitions: 5.516632318496704


In [0]:
spark.conf.set("spark.sql.shuffle.partitions", 800)

start = time.time()

train_df.groupBy("District").count().collect()

end = time.time()
print("800 partitions:", end - start)

800 partitions: 5.395401239395142


In [0]:
small_df = train_df.sample(fraction=0.25, seed=42)
medium_df = train_df.sample(fraction=0.5, seed=42)
large_df = train_df

In [0]:
spark.conf.set("spark.sql.shuffle.partitions", 200)

import time
start = time.time()

small_df.groupBy("District").count().collect()

end = time.time()
print("Small dataset:", end - start)

Small dataset: 5.4515204429626465


In [0]:
spark.conf.set("spark.sql.shuffle.partitions", 400)

start = time.time()

medium_df.groupBy("District").count().collect()

end = time.time()
print("Medium dataset:", end - start)

Medium dataset: 5.3999552726745605


In [0]:
spark.conf.set("spark.sql.shuffle.partitions", 800)

start = time.time()

large_df.groupBy("District").count().collect()

end = time.time()
print("Large dataset:", end - start)

Large dataset: 5.152501344680786


In [0]:
district_lookup = crime_df.select("District").distinct()

district_lookup = district_lookup.withColumn(
    "District_Category",
    when(col("District") <= 10, "Low_Number")
    .otherwise("High_Number")
)

In [0]:
import time

start = time.time()

normal_join = crime_df.join(
    district_lookup,
    on="District",
    how="left"
)

normal_join.count()

end = time.time()
print("Normal Join Time:", end - start)

Normal Join Time: 1.1431291103363037


In [0]:
from pyspark.sql.functions import broadcast

start = time.time()

broadcast_join = crime_df.join(
    broadcast(district_lookup),
    on="District",
    how="left"
)

broadcast_join.count()

end = time.time()
print("Broadcast Join Time:", end - start)

Broadcast Join Time: 0.47626447677612305


In [0]:
import pandas as pd

# Take small sample (important for memory)
sample_df = crime_ml_df.sample(fraction=0.05, seed=42).toPandas()

# Separate features and label
X = sample_df.drop("label", axis=1)
y = sample_df["label"]

# Convert categorical columns
X = pd.get_dummies(X)

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

gb = GradientBoostingClassifier()
gb.fit(X_train, y_train)

y_pred = gb.predict_proba(X_test)[:,1]
auc_sklearn = roc_auc_score(y_test, y_pred)

print("Scikit-learn Gradient Boosting AUC:", auc_sklearn)

Scikit-learn Gradient Boosting AUC: 0.8457563792224077


In [0]:
results = {
    "Logistic Regression (Spark)": auc_lr,
    "Decision Tree (Spark)": auc_dt,
    "Random Forest (Spark)": auc_rf,
    "Gradient Boosting (Sklearn)": auc_sklearn
}

for model, score in results.items():
    print(f"{model}: {score:.4f}")

Logistic Regression (Spark): 0.7820
Decision Tree (Spark): 0.4071
Random Forest (Spark): 0.7834
Gradient Boosting (Sklearn): 0.8458


In [0]:
predictions_rf.groupBy("label", "prediction").count().show()

+-----+----------+------+
|label|prediction| count|
+-----+----------+------+
|    1|       0.0| 84684|
|    0|       0.0|860921|
|    1|       1.0| 53409|
|    0|       1.0| 25940|
+-----+----------+------+



In [0]:
print("LR:", auc_lr)
print("DT:", auc_dt)
print("SK:", auc_sklearn)

LR: 0.781963544237379
DT: 0.4070756792484828
SK: 0.8457563792224077


In [0]:
auc_rf = 0.78   

In [0]:
results = {
    "Logistic Regression (Spark)": auc_lr,
    "Decision Tree (Spark)": auc_dt,
    "Random Forest (Spark)": auc_rf,
    "Gradient Boosting (Sklearn)": auc_sklearn
}

for model, score in results.items():
    print(f"{model}: {score:.4f}")

Logistic Regression (Spark): 0.7820
Decision Tree (Spark): 0.4071
Random Forest (Spark): 0.7800
Gradient Boosting (Sklearn): 0.8458


In [0]:
from pyspark.sql import Row

cm_data = [
    Row(Actual="Positive (1)", Predicted="Positive (1)", Count=53409),   
    Row(Actual="Positive (1)", Predicted="Negative (0)", Count=84684),   
    Row(Actual="Negative (0)", Predicted="Positive (1)", Count=25940),  
    Row(Actual="Negative (0)", Predicted="Negative (0)", Count=860921)   
]

cm_df = spark.createDataFrame(cm_data)

cm_df.show()

+------------+------------+------+
|      Actual|   Predicted| Count|
+------------+------------+------+
|Positive (1)|Positive (1)| 53409|
|Positive (1)|Negative (0)| 84684|
|Negative (0)|Positive (1)| 25940|
|Negative (0)|Negative (0)|860921|
+------------+------------+------+



In [0]:
import pandas as pd

scalability_data = pd.DataFrame({
    "Setting": [
        "200 Partitions",
        "400 Partitions",
        "800 Partitions",
        "Small Dataset",
        "Medium Dataset",
        "Large Dataset"
    ],
    "Category": [
        "Partitions",
        "Partitions",
        "Partitions",
        "Dataset Size",
        "Dataset Size",
        "Dataset Size"
    ],
    "Time_sec": [
        5.38,
        5.51,
        5.39,
        5.45,
        5.40,
        5.15
    ]
})

scalability_data.to_csv("/tmp/scalability_detailed.csv", index=False)

In [0]:
import os
os.listdir("/tmp")

['keyutil_spark.host.local_7547850219144034258.key',
 'scalability_detailed.csv',
 'driver-daemon.pid',
 'chauffeur-daemon.pid',
 'driver-daemon-params',
 'clean_up_local_disk.sh',
 'driver-env.sh',
 'tmpst1dc4ij',
 'matplotlib-root',
 'chauffeur-daemon-params',
 'prefetch_critical_files.sh',
 'pre_start.sh',
 'databricks-jni9015550340383746858',
 'dbr_entry_point.py',
 'master-params',
 'dbr.conf',
 'keyutil_spark.host.local_15594263997167852236.crt',
 'chauffeur-env.sh',
 'validate_container_security.py',
 'dbr-consolidated-secret-conf-envvars',
 'custom-spark.conf',
 'python_lsp_logs',
 'hsperfdata_root']

In [0]:
import pandas as pd

scalability_data = pd.DataFrame({
    "Setting": [
        "200 Partitions",
        "400 Partitions",
        "800 Partitions",
        "Small Dataset",
        "Medium Dataset",
        "Large Dataset"
    ],
    "Category": [
        "Partitions",
        "Partitions",
        "Partitions",
        "Dataset Size",
        "Dataset Size",
        "Dataset Size"
    ],
    "Time_sec": [
        5.38,
        5.51,
        5.39,
        5.45,
        5.40,
        5.15
    ]
})

scalability_data

,Setting,Category,Time_sec
0,200 Partitions,Partitions,5.38
1,400 Partitions,Partitions,5.51
2,800 Partitions,Partitions,5.39
3,Small Dataset,Dataset Size,5.45
4,Medium Dataset,Dataset Size,5.40
5,Large Dataset,Dataset Size,5.15


In [0]:
train_df = crime_ml_df.filter(col("Year") < 2022)

In [0]:
train_df.groupBy("District").count().show()

+--------+------+
|District| count|
+--------+------+
|       5|332480|
|      12|367018|
|      22|245196|
|       2|353076|
|      10|322597|
|      11|482733|
|      31|   222|
|      18|333316|
|      15|323295|
|      16|248802|
|       7|437106|
|       1|297806|
|      17|215600|
|       8|503967|
|      19|333759|
|      21|     4|
|       6|436160|
|      20|131193|
|       9|367072|
|      25|427453|
+--------+------+
only showing top 20 rows


In [0]:
train_df.groupBy("District").count().explain(True)

== Parsed Logical Plan ==
'Aggregate ['District], ['District, 'count(1) AS count#13330]
+- 'Filter '`<`('Year, 2022)
   +- 'UnresolvedRelation [crime_parquet_table], [], false

== Analyzed Logical Plan ==
District: int, count: bigint
Aggregate [District#13364], [District#13364, count(1) AS count#13330L]
+- Filter (Year#13370 < 2022)
   +- SubqueryAlias workspace.crime_data.crime_parquet_table
      +- Relation workspace.crime_data.crime_parquet_table[ID#13353,Case_Number#13354,Date#13355,Block#13356,IUCR#13357,Primary_Type#13358,Description#13359,Location_Description#13360,Arrest#13361,Domestic#13362,Beat#13363,District#13364,Ward#13365,Community_Area#13366,FBI_Code#13367,X_Coordinate#13368,Y_Coordinate#13369,Year#13370,Updated_On#13371,Latitude#13372,Longitude#13373,Location#13374] parquet

== Optimized Logical Plan ==
Aggregate [District#13364], [District#13364, count(1) AS count#13330L]
+- Project [District#13364]
   +- Filter (isnotnull(Year#13370) AND (Year#13370 < 2022))
      +-